# 🏖️ Personalized Holiday Management Agent

An autonomous, multi-agent AI system for intelligent travel planning.

---

## 📋 API Keys Required

Before running this notebook, add the following secrets in **Colab → Secrets (🔑)**:

| Secret Name | Description | Where to Get |
|---|---|---|
| `GROQ_API_KEY` | LLM inference (llama-instant) | https://console.groq.com |
| `MONGO_DB_URL` | MongoDB Atlas connection string | https://cloud.mongodb.com |
| `MLFLOW_TRACKING_URI` | DagsHub MLflow URI | https://dagshub.com |
| `MLFLOW_TRACKING_USERNAME` | DagsHub username | https://dagshub.com |
| `MLFLOW_TRACKING_PASSWORD` | DagsHub token/password | https://dagshub.com |
| `MEM0_API_KEY` *(optional)* | Mem0 cloud memory (if using cloud) | https://app.mem0.ai |

---

## 🏗️ Project Structure

```
holiday_management/
├── config/
│   └── settings.py          # Centralised config & env vars
├── agents/
│   ├── __init__.py
│   ├── planner.py           # Agent 1: Strategy / Skeleton Itinerary
│   └── researcher.py        # Agent 2: Fact Verification
├── teams/
│   ├── __init__.py
│   └── holiday_team.py      # Orchestrates agent hand-offs
└── utils/
    ├── __init__.py
    ├── state.py             # Shared state object between agents
    └── utils.py             # Helper functions

all-utils/
└── utilities/
    ├── pydantic_models.py          # Phase 1: Request/Response schemas
    ├── query_validation_transformation.py  # Phase 2: Query cleaning
    ├── logging_example.py          # Phase 3: Structured logging
    └── mem0_example.py             # Phase 4: Long-term memory
```

---

## 🔄 Workflow Phases

| Phase | Title | Description |
|---|---|---|
| 0 | Environment Setup | Install packages, set secrets, configure MLflow |
| 1 | Pydantic Models | Request/Response schema validation |
| 2 | Query Validation & Transformation | Clean & normalize user queries |
| 3 | Structured Logging | App-wide logging utility |
| 4 | Mem0 Memory | Long-term user preference memory |
| 5 | Project Structure | Build holiday_management package |
| 6 | Agent Implementation | Planner & Researcher agents (Groq LLM) |
| 7 | Team Orchestration | Run the full multi-agent pipeline |
| 8 | MLflow Experiment Tracking | Log runs, metrics, artifacts |
| 9 | CI/CD & Git Push | Push code to GitHub |


---
## ⚙️ Phase 0 — Environment Setup

Install all required packages and configure secrets from Colab's secret manager.


In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# PHASE 0 — Package Installation
# ─────────────────────────────────────────────────────────────────────────────
# Core dependencies:
#   • groq            — Groq Cloud SDK for llama-instant LLM inference
#   • langchain ≥1.2  — LLM chaining framework (user requirement)
#   • mem0ai          — Long-term user memory backed by ChromaDB
#   • pydantic ≥2     — Data validation and structured output
#   • mlflow          — Experiment tracking and artifact logging
#   • pymongo         — MongoDB driver for persistent storage
#   • python-dotenv   — Load .env files (fallback to os.environ)

%pip install -q \
    groq \
    "langchain" \
    "langchain-groq"\
    "langchain-community" \
    mem0ai \
    chromadb \
    "pydantic[email]" \
    mlflow \
    pymongo \
    python-dotenv \
    dagshub \
    sentence-transformers

print("✅ All packages installed successfully.")

✅ All packages installed successfully.


In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# PHASE 0 — Secrets & Environment Variables
# ─────────────────────────────────────────────────────────────────────────────
# Reads all required API keys from Colab's built-in secret manager
# (Runtime → Manage Secrets) instead of hard-coding them.
#
# NOTE: Enable notebook access for each secret in the Colab Secrets sidebar.

import os
from google.colab import userdata

# ── Groq LLM ─────────────────────────────────────────────────────────────────
os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')

# ── MongoDB (from Colab Secrets) ──────────────────────────────────────────────
os.environ['MONGO_DB_URL'] = userdata.get('MONGO_DB_URL')

# ── MLflow / DagsHub ──────────────────────────────────────────────────────────
USE_DAGSHUB = True  # ✅ Set False to log locally inside Colab

if USE_DAGSHUB:
    os.environ['MLFLOW_TRACKING_URI']      = userdata.get('MLFLOW_TRACKING_URI')
    os.environ['MLFLOW_TRACKING_USERNAME'] = userdata.get('MLFLOW_TRACKING_USERNAME')
    os.environ['MLFLOW_TRACKING_PASSWORD'] = userdata.get('MLFLOW_TRACKING_PASSWORD')
else:
    # Local MLflow — logs saved inside Colab runtime
    os.environ['MLFLOW_TRACKING_URI'] = f"file://{os.getcwd()}/mlruns"

print('✅ Env vars set.')
print(f"   MLFLOW_TRACKING_URI = {os.environ['MLFLOW_TRACKING_URI']}")

✅ Env vars set.
   MLFLOW_TRACKING_URI = https://dagshub.com/prithusarkar90/networksecurity.mlflow


In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# PHASE 0 — Directory Structure Initialisation
# ─────────────────────────────────────────────────────────────────────────────
# Creates the complete project folder hierarchy expected by the package.
# Mirrors the original project layout (excluding Docker-specific files).

import os
from pathlib import Path

# All directories the project requires
dirs = [
    "holiday_management/config",
    "holiday_management/agents",
    "holiday_management/teams",
    "holiday_management/utils",
    "all-utils/utilities",
    "all-utils/db",
    "phase_outputs",       # ← saves per-phase artefacts
    "static",
]

for d in dirs:
    Path(d).mkdir(parents=True, exist_ok=True)

print("✅ Directory structure created:")
for d in dirs:
    print(f"   {d}/")

✅ Directory structure created:
   holiday_management/config/
   holiday_management/agents/
   holiday_management/teams/
   holiday_management/utils/
   all-utils/utilities/
   all-utils/db/
   phase_outputs/
   static/


---
## 📐 Phase 1 — Pydantic Request & Response Models

Pydantic is the bridge between messy human/LLM text and structured Python objects.  
It guarantees that every request entering the system is valid **before** hitting the LLM.

### Why this matters for GenAI:
- **Structured Output** — forces the LLM to return JSON that matches a schema
- **Tool Use** — LLMs see the schema and know exactly how to call your functions
- **Hallucination Prevention** — invalid AI output raises a `ValidationError` instantly
- **Auto-Documentation** — FastAPI reads Pydantic models and generates Swagger docs


In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# PHASE 1 — Pydantic Models  (all-utils/utilities/pydantic_models.py)
# ─────────────────────────────────────────────────────────────────────────────

PYDANTIC_CODE = '''
# pydantic_models.py
# Defines the validated input/output schemas for the Holiday Management Agent.
# Every request flowing into the system is checked against SearchRequest.
# Every response returned is wrapped in SearchResponse.

from pydantic import BaseModel, EmailStr, Field, field_validator
from datetime import datetime
from typing import Optional, List


# ── Input Schema ──────────────────────────────────────────────────────────────
class SearchRequest(BaseModel):
    """Validated request model for any user search/query coming into the system."""

    user_id: str = Field(..., min_length=3, max_length=50,
                         description="Unique identifier for the user.")
    email: EmailStr  # pydantic automatically validates email format
    query: str = Field(..., min_length=1, max_length=200,
                       description="The user's natural language query.")
    tags: Optional[List[str]] = Field(default_factory=list,
                                      description="Optional topic tags.")

    @field_validator('query')
    def query_must_not_be_empty(cls, value: str) -> str:
        """Strip whitespace and reject blank queries early."""
        if not value.strip():
            raise ValueError('Query must not be empty or whitespace')
        return value.strip()


# ── Output Schema ─────────────────────────────────────────────────────────────
class SearchResponse(BaseModel):
    """Validated response model returned after processing a user query."""

    status: str
    message: str
    result_count: int = Field(0, ge=0,
                              description="Number of results returned (>= 0).")
    results: List[dict] = Field(default_factory=list)
    processed_at: datetime = Field(default_factory=datetime.utcnow,
                                   description="UTC timestamp of processing.")


# ── Holiday-specific Trip Request ─────────────────────────────────────────────
class TripPlanRequest(BaseModel):
    """Schema for a holiday planning request sent to the agent team."""

    user_id: str = Field(..., min_length=3)
    destination: str = Field(..., min_length=2,
                             description="Target travel destination.")
    duration_days: int = Field(..., ge=1, le=30,
                               description="Number of days for the trip.")
    interests: Optional[List[str]] = Field(default_factory=list,
                                           description="User interests, e.g. food, culture.")
    budget: Optional[str] = Field(None,
                                  description="Budget level: low / medium / high.")

    @field_validator('budget')
    def budget_must_be_valid(cls, v):
        if v and v.lower() not in {"low", "medium", "high"}:
            raise ValueError("budget must be 'low', 'medium', or 'high'")
        return v.lower() if v else v


# ── Helper ────────────────────────────────────────────────────────────────────
def build_search_response(request: SearchRequest) -> SearchResponse:
    """Simulates building a search response for a validated request."""
    example_results = [
        {"id": 1, "title": "Example item", "query": request.query},
    ]
    return SearchResponse(
        status="success",
        message=f"Search completed for user {request.user_id}",
        result_count=len(example_results),
        results=example_results,
    )


def demo() -> None:
    """Run a quick validation demo."""
    request_payload = {
        "user_id": "user123",
        "email": "user@example.com",
        "query": "plan a 5-day trip to Paris",
        "tags": ["travel", "europe"],
    }
    request = SearchRequest(**request_payload)
    response = build_search_response(request)

    print("Request model:")
    print(request.model_dump_json(indent=2))
    print()
    print("Response model:")
    print(response.model_dump_json(indent=2))

    # Also test the TripPlanRequest
    trip = TripPlanRequest(
        user_id="user123",
        destination="Tokyo",
        duration_days=7,
        interests=["anime", "food", "temples"],
        budget="medium"
    )
    print()
    print("Trip Plan Request:")
    print(trip.model_dump_json(indent=2))
'''

# ── Write file to disk ────────────────────────────────────────────────────────
with open("all-utils/utilities/pydantic_models.py", "w") as f:
    f.write(PYDANTIC_CODE.strip())

print("✅ pydantic_models.py written to all-utils/utilities/")

✅ pydantic_models.py written to all-utils/utilities/


In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# PHASE 1 — Run & Save Output
# ─────────────────────────────────────────────────────────────────────────────

import sys, json
from io import StringIO

# Dynamically load and execute the module we just wrote
sys.path.insert(0, 'all-utils')
from utilities.pydantic_models import demo, SearchRequest, TripPlanRequest

# Capture printed output
old_stdout = sys.stdout
sys.stdout = buf = StringIO()
demo()
sys.stdout = old_stdout
phase1_output = buf.getvalue()

print(phase1_output)

# Save phase output to disk
with open("phase_outputs/phase1_pydantic_output.txt", "w") as f:
    f.write(phase1_output)

print("\n✅ Phase 1 output saved → phase_outputs/phase1_pydantic_output.txt")

Request model:
{
  "user_id": "user123",
  "email": "user@example.com",
  "query": "plan a 5-day trip to Paris",
  "tags": [
    "travel",
    "europe"
  ]
}

Response model:
{
  "status": "success",
  "message": "Search completed for user user123",
  "result_count": 1,
  "results": [
    {
      "id": 1,
      "title": "Example item",
      "query": "plan a 5-day trip to Paris"
    }
  ],
  "processed_at": "2026-05-08T13:32:53.684839"
}

Trip Plan Request:
{
  "user_id": "user123",
  "destination": "Tokyo",
  "duration_days": 7,
  "interests": [
    "anime",
    "food",
    "temples"
  ],
  "budget": "medium"
}


✅ Phase 1 output saved → phase_outputs/phase1_pydantic_output.txt


/usr/local/lib/python3.12/dist-packages/pydantic/main.py:250: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  validated_self = self.__pydantic_validator__.validate_python(data, self_instance=self)


---
## 🔍 Phase 2 — Query Validation & Transformation

Before sending user queries to the LLM, we clean and normalize them to:
- **Boost RAG retrieval accuracy** — stop-words hurt vector search
- **Save LLM tokens** — shorter queries cost less per API call  
- **Enable semantic caching** — synonymized queries map to the same cache key
- **Prevent prompt injection** — block invalid characters early


In [7]:
# ─────────────────────────────────────────────────────────────────────────────
# PHASE 2 — Query Validation & Transformation
#           (all-utils/utilities/query_validation_transformation.py)
# ─────────────────────────────────────────────────────────────────────────────

QUERY_CODE = '''
# query_validation_transformation.py
# Cleans and normalizes raw user queries before they reach the LLM.
#
# Pipeline:
#   1. validate_query()   — reject short / malformed queries immediately
#   2. transform_query()  — lowercase, collapsed whitespace, strip stop-words,
#                           expand synonyms, and produce a cache-friendly
#                           signature string.

import re
from typing import Dict


# Only alphanumeric + a safe set of punctuation is permitted.
# This blocks SQL injection, HTML tags, and unusual Unicode.
ALLOWED_QUERY_PATTERN = re.compile(r"""^[a-zA-Z0-9\s?@#\-_.,'"()]+$""")

# Common English stop-words that add noise to keyword searches.
STOP_WORDS = {"the", "is", "and", "or", "for", "a", "an", "to", "i", "want"}

# Synonym map: map user vocabulary → canonical vocabulary.
# Extend this dict to cover your domain (e.g. travel-specific terms).
SYNONYMS = {
    "buy":    "purchase",
    "find":   "search",
    "latest": "recent",
    "trip":   "travel",
    "holiday": "vacation",
    "book":   "reserve",
    "cheap":  "budget",
}


def validate_query(query: str) -> bool:
    """Raises ValueError if the query is too short or contains illegal chars."""
    if not query or len(query.strip()) < 3:
        raise ValueError("Query must be at least 3 characters long.")
    if not ALLOWED_QUERY_PATTERN.match(query):
        raise ValueError("Query contains invalid characters.")
    return True


def transform_query(query: str) -> Dict[str, str]:
    """
    Returns a dict with four representations of the query:
      original  — raw input from the user
      normalized — lowercase, collapsed whitespace
      cleaned   — stop-words removed, synonyms applied
      signature — cleaned query joined with underscores (cache key)
    """
    # Step 1: lowercase and collapse extra whitespace
    normalized = query.strip().lower()
    normalized = re.sub(r"\\s+", " ", normalized)

    # Step 2: tokenise, remove stop-words, apply synonym map
    tokens = normalized.split()
    tokens = [SYNONYMS.get(token, token) for token in tokens
              if token not in STOP_WORDS]
    cleaned_query = " ".join(tokens)

    # Step 3: produce a deterministic cache/signature key
    query_signature = cleaned_query.replace(" ", "_")

    return {
        "original":   query,
        "normalized": normalized,
        "cleaned":    cleaned_query,
        "signature":  query_signature,
    }


def handle_query(query: str) -> Dict[str, str]:
    """Full pipeline: validate then transform."""
    validate_query(query)
    return transform_query(query)
'''

with open("all-utils/utilities/query_validation_transformation.py", "w") as f:
    f.write(QUERY_CODE.strip())

print("✅ query_validation_transformation.py written")

✅ query_validation_transformation.py written


<>:22: SyntaxWarning: invalid escape sequence '\s'
<>:22: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_64891/1749059932.py:22: SyntaxWarning: invalid escape sequence '\s'
  ALLOWED_QUERY_PATTERN = re.compile(r"""^[a-zA-Z0-9\s?@#\-_.,'"()]+$""")


In [8]:
# ─────────────────────────────────────────────────────────────────────────────
# PHASE 2 — Run & Save Output
# ─────────────────────────────────────────────────────────────────────────────

from utilities.query_validation_transformation import handle_query
import json

# Test queries — including travel-specific phrasing
test_queries = [
    "I want to find a cheap trip to Paris for 5 days",
    "latest hotels in Tokyo for a holiday",
    "Where can I buy tickets to the Eiffel Tower?",
]

results = []
for q in test_queries:
    try:
        result = handle_query(q)
        results.append(result)
        print(f"✅ Processed: {q}")
        print(json.dumps(result, indent=2))
        print()
    except ValueError as e:
        print(f"❌ Validation failed: {e}")

# Save outputs
with open("phase_outputs/phase2_query_transform_output.json", "w") as f:
    json.dump(results, f, indent=2)

print("✅ Phase 2 output saved → phase_outputs/phase2_query_transform_output.json")

✅ Processed: I want to find a cheap trip to Paris for 5 days
{
  "original": "I want to find a cheap trip to Paris for 5 days",
  "normalized": "i want to find a cheap trip to paris for 5 days",
  "cleaned": "search budget travel paris 5 days",
  "signature": "search_budget_travel_paris_5_days"
}

✅ Processed: latest hotels in Tokyo for a holiday
{
  "original": "latest hotels in Tokyo for a holiday",
  "normalized": "latest hotels in tokyo for a holiday",
  "cleaned": "recent hotels in tokyo vacation",
  "signature": "recent_hotels_in_tokyo_vacation"
}

✅ Processed: Where can I buy tickets to the Eiffel Tower?
{
  "original": "Where can I buy tickets to the Eiffel Tower?",
  "normalized": "where can i buy tickets to the eiffel tower?",
  "cleaned": "where can purchase tickets eiffel tower?",
  "signature": "where_can_purchase_tickets_eiffel_tower?"
}

✅ Phase 2 output saved → phase_outputs/phase2_query_transform_output.json


---
## 📋 Phase 3 — Structured Logging

GenAI apps are "black boxes" — you must log everything to debug them.  
This phase sets up a dual-output logger (console + file) that captures:
- LLM inputs / outputs
- API latency timings
- Chain step failures
- Cost / token usage


In [9]:
# ─────────────────────────────────────────────────────────────────────────────
# PHASE 3 — Structured Logging  (all-utils/utilities/logging_example.py)
# ─────────────────────────────────────────────────────────────────────────────

LOGGING_CODE = '''
# logging_example.py
# Provides a reusable, dual-output logger for the Holiday Management Agent.
#
# Two handlers are attached to every logger instance:
#   • Console (INFO+) — coloured output in Colab / terminal
#   • File (DEBUG+)   — full trace written to utility_logging_example.log
#
# Usage:
#   from utilities.logging_example import get_app_logger
#   logger = get_app_logger("my_module")
#   logger.info("Agent started.")

import logging
import sys


def get_app_logger(name: str = __name__) -> logging.Logger:
    """
    Return a named logger with console + file handlers.
    Calling this multiple times with the same name is safe — handlers
    are only added once (guard via `logger.handlers`).
    """
    logger = logging.getLogger(name)
    if logger.handlers:        # already configured — return as-is
        return logger

    logger.setLevel(logging.DEBUG)   # capture everything at the root level

    # ── Console handler: INFO and above ──────────────────────────────────────
    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setLevel(logging.INFO)
    console_handler.setFormatter(logging.Formatter(
        "%(asctime)s - %(name)s - %(levelname)s - %(message)s"
    ))

    # ── File handler: DEBUG and above (full audit trail) ─────────────────────
    file_handler = logging.FileHandler(
        "utility_logging_example.log", encoding="utf-8"
    )
    file_handler.setLevel(logging.DEBUG)
    file_handler.setFormatter(logging.Formatter(
        "%(asctime)s | %(levelname)s | %(name)s | %(message)s"
    ))

    logger.addHandler(console_handler)
    logger.addHandler(file_handler)
    logger.propagate = False  # prevent double-logging via root logger
    return logger


def run_logging_demo() -> None:
    """Demonstrates all log levels including exception capture."""
    logger = get_app_logger("utility_logger")

    logger.debug("Debugging values: %s", {"step": 1, "status": "starting"})
    logger.info("Holiday Agent logging demo started.")
    logger.warning("Token budget is approaching 80%% — consider truncating context.")

    try:
        value = 10 / 0   # simulate a runtime error in an agent step
    except ZeroDivisionError:
        logger.exception("Division error — would be an API/timeout failure in production.")

    logger.info("Logging demo finished.")
'''

with open("all-utils/utilities/logging_example.py", "w") as f:
    f.write(LOGGING_CODE.strip())

print("✅ logging_example.py written")

✅ logging_example.py written


In [10]:
# ─────────────────────────────────────────────────────────────────────────────
# PHASE 3 — Run & Save Output
# ─────────────────────────────────────────────────────────────────────────────

# Reload the module (it may have been imported before we rewrote it)
import importlib
import utilities.logging_example as log_mod
importlib.reload(log_mod)

log_mod.run_logging_demo()

# Copy the log file to phase_outputs for archiving
import shutil
shutil.copy("utility_logging_example.log", "phase_outputs/phase3_logging_output.log")

print("\n✅ Phase 3 output saved → phase_outputs/phase3_logging_output.log")

2026-05-08 13:33:07,143 - utility_logger - INFO - Holiday Agent logging demo started.
2026-05-08 13:33:07,145 - utility_logger - WARNING - Token budget is approaching 80%% — consider truncating context.
2026-05-08 13:33:07,146 - utility_logger - ERROR - Division error — would be an API/timeout failure in production.
Traceback (most recent call last):
  File "/content/all-utils/utilities/logging_example.py", line 60, in run_logging_demo
    value = 10 / 0   # simulate a runtime error in an agent step
            ~~~^~~
ZeroDivisionError: division by zero
2026-05-08 13:33:07,148 - utility_logger - INFO - Logging demo finished.

✅ Phase 3 output saved → phase_outputs/phase3_logging_output.log


---
## 🧠 Phase 4 — Mem0 Long-Term Memory

Standard LLMs are stateless — they forget everything between sessions.  
Mem0 adds a **persistent, evolving memory layer** backed by ChromaDB (local vector store).

This enables:
- **Personalization** — remember user preferences across sessions
- **Conflict resolution** — the LLM detects when new info supersedes old
- **Semantic search** — find relevant memories by meaning, not keyword
- **GDPR audit trail** — `.history()` shows every memory change with timestamps


In [11]:
# ─────────────────────────────────────────────────────────────────────────────
# PHASE 4 — Mem0 Memory  (all-utils/utilities/mem0_example.py)
# ─────────────────────────────────────────────────────────────────────────────

MEM0_CODE = '''
# mem0_example.py
# Demonstrates long-term, evolving user memory for the Holiday Management Agent.
#
# Uses:
#   • ChromaDB (local ./all-utils/db) as the vector store
#   • Groq llama-instant-1.5b as the LLM for memory reasoning
#     (replaces the original OpenAI gpt-4o-mini to avoid OpenAI dependency)
#
# Flow:
#   1. Initialise Memory with Groq + Chroma config
#   2. Store initial preference (e.g. "I prefer beach holidays")
#   3. Update preference (user changes their mind)
#   4. Print observability history — shows the OLD → NEW evolution
#   5. Semantic search — retrieve the most relevant memory

import os

# Remove the SSL cert override that can break Colab network calls
if "SSL_CERT_FILE" in os.environ:
    del os.environ["SSL_CERT_FILE"]

from mem0 import Memory


def get_mem0_config(db_path: str = "all-utils/db") -> dict:
    """
    Build the Mem0 config dict.
    Uses Groq as the LLM provider so no OpenAI key is required.
    """
    return {
        "vector_store": {
            "provider": "chroma",
            "config": {
                "collection_name": "holiday_agent_memory",
                "path": db_path
            }
        },
        "llm": {
            "provider": "groq",
            "config": {
                "model": "llama-3.1-8b-instant",   # fast & cost-effective
                "temperature": 0,                   # deterministic reasoning
                "max_tokens": 256,                  # keep memory calls cheap
                "api_key": os.environ.get("GROQ_API_KEY")
            }
        },
        "embedder": {
            "provider": "huggingface",
            "config": {
                "model": "all-MiniLM-L6-v2"
            }
        }
    }


def run_observability_demo(user_id: str = "traveller_001") -> None:
    """
    End-to-end Mem0 demo: add → update → history → search.
    """

    config = get_mem0_config()
    m = None
    try:
        print("Attempting to initialize Mem0...")
        m = Memory.from_config(config)
        print("Mem0 initialized successfully.")
    except Exception as e:
        print(f"Error initializing Mem0: {e}")
        return # Exit if Mem0 cannot be initialized

    # ── Step 1: Store initial travel preference ───────────────────────────────
    print()
    print("--- [Step 1] Storing Initial Preference ---")
    result = m.add(
        "I prefer beach holidays and warm weather destinations.",
        user_id=user_id
    )

    # Safely extract memory ID (API returns list or dict depending on version)
    mem_id = None
    if isinstance(result, list) and result:
        mem_id = result[0].get("id")
    elif isinstance(result, dict):
        res_list = result.get("results") or result.get("memories") or []
        if res_list:
            mem_id = res_list[0].get("id")
    print(f"   Memory ID captured: {mem_id}")

    # ── Step 2: User changes their mind ──────────────────────────────────────
    print("--- [Step 2] Updating Preference ---")
    m.add(
        "Actually, I have changed my mind. I now prefer mountain hiking and cold weather.",
        user_id=user_id
    )

    # ── Step 3: Observability — show memory evolution ─────────────────────────
    print()
    print("--- [Step 3] Observability Report: Memory Evolution ---")
    if mem_id:
        history = m.history(memory_id=mem_id)
        for entry in history:
            print(f"  Event : {entry.get('event')}")
            old = entry.get('old_memory') or entry.get('old_value') or "(initial)"
            new = entry.get('memory') or entry.get('new_value')
            print(f"  Old   : {old}")
            print(f"  New   : {new}")
            print("  " + "-" * 40)
    else:
        print("  Note: Memory ID not captured — check db folder.")

    # ── Step 4: Semantic search for relevant memories ─────────────────────────
    print()
    print("--- [Step 4] Semantic Memory Search ---")
    search_results = m.search(
        "What kind of holiday destination suits this user?",
        filters={"user_id": user_id}
    )

    memories = (
        search_results.get("results")
        if isinstance(search_results, dict)
        else search_results
    )

    if memories:
        for res in memories:
            val = res.get("memory") or res.get("payload", {}).get("value")
            score = res.get("score", "N/A")
            print(f"  Memory: {val}  (relevance score: {score})")
    else:
        print("  No memories found for this user.")
'''

with open("all-utils/utilities/mem0_example.py", "w") as f:
    f.write(MEM0_CODE.strip())

print("✅ mem0_example.py written (Groq-powered, no OpenAI dependency)")

✅ mem0_example.py written (Groq-powered, no OpenAI dependency)


In [13]:
# ─────────────────────────────────────────────────────────────────────────────
# PHASE 4 — Run Mem0 Demo & Save Output
# ─────────────────────────────────────────────────────────────────────────────

import sys
from io import StringIO
import importlib
import shutil
import os

# Ensure the module is fully reloaded by removing it from sys.modules if present
if 'utilities.mem0_example' in sys.modules:
    del sys.modules['utilities.mem0_example']

# Clear ChromaDB directory before running Mem0 demo
chroma_db_path = "all-utils/db"
if os.path.exists(chroma_db_path):
    shutil.rmtree(chroma_db_path)
    print(f"✅ Cleared existing ChromaDB at {chroma_db_path}")

import utilities.mem0_example as mem_mod
# importlib.reload(mem_mod) # Reload not strictly needed after del and re-import

old_stdout = sys.stdout
sys.stdout = buf = StringIO()

mem_mod.run_observability_demo(user_id="traveller_001")

sys.stdout = old_stdout
phase4_output = buf.getvalue()
print(phase4_output)

with open("phase_outputs/phase4_mem0_output.txt", "w") as f:
    f.write(phase4_output)

print("\n✅ Phase 4 output saved → phase_outputs/phase4_mem0_output.txt")

✅ Cleared existing ChromaDB at all-utils/db


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/usr/local/lib/python3.12/dist-packages/mem0/embeddings/huggingface.py:27: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.config.embedding_dims = self.config.embedding_dims or self.model.get_sentence_embedding_dimension()
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
ERROR:mem0.memory.main:LLM extraction failed: Error code: 413 - {'error'

Attempting to initialize Mem0...
Mem0 initialized successfully.

--- [Step 1] Storing Initial Preference ---
   Memory ID captured: None
--- [Step 2] Updating Preference ---

--- [Step 3] Observability Report: Memory Evolution ---
  Note: Memory ID not captured — check db folder.

--- [Step 4] Semantic Memory Search ---
  No memories found for this user.


✅ Phase 4 output saved → phase_outputs/phase4_mem0_output.txt


---
## 📦 Phase 5 — Build the `holiday_management` Package

Creates the installable Python package with:
- `config/settings.py` — centralised environment config
- `utils/state.py` — shared State object passed between agents  
- `utils/utils.py` — helper functions
- `agents/planner.py` — Agent 1: strategy skeleton
- `agents/researcher.py` — Agent 2: fact verification
- `teams/holiday_team.py` — orchestrates agent hand-offs


In [14]:
# ─────────────────────────────────────────────────────────────────────────────
# PHASE 5a — Config & Settings  (holiday_management/config/settings.py)
# ─────────────────────────────────────────────────────────────────────────────

# Create all __init__.py files first
for pkg in ["holiday_management", "holiday_management/config",
            "holiday_management/agents", "holiday_management/teams",
            "holiday_management/utils"]:
    open(f"{pkg}/__init__.py", "a").close()

SETTINGS_CODE = '''
# settings.py
# Single source of truth for all environment variables and runtime settings.
# Reads from os.environ (populated by Colab Secrets in Phase 0).

import os


class Settings:
    """
    Centralised application settings.
    Every agent and utility imports from here — no scattered os.getenv calls.
    """

    # ── LLM ──────────────────────────────────────────────────────────────────
    # Using Groq llama-3.1-8b-instant: fast inference, generous free tier.
    GROQ_API_KEY:   str = os.environ.get("GROQ_API_KEY", "")
    LLM_MODEL:      str = "llama-3.1-8b-instant"   # versatile & within token limits
    LLM_MAX_TOKENS: int = 256                      # safe ceiling for Groq free tier
    LLM_TEMPERATURE: float = 0.3                    # slightly creative but stable

    # ── Databases ────────────────────────────────────────────────────────────
    MONGO_DB_URL:   str = os.environ.get("MONGO_DB_URL", "")
    CHROMA_DB_PATH: str = "all-utils/db"            # local ChromaDB for Mem0

    # ── MLflow ───────────────────────────────────────────────────────────────
    MLFLOW_TRACKING_URI:      str = os.environ.get("MLFLOW_TRACKING_URI", "./mlruns")
    MLFLOW_TRACKING_USERNAME: str = os.environ.get("MLFLOW_TRACKING_USERNAME", "")
    MLFLOW_TRACKING_PASSWORD: str = os.environ.get("MLFLOW_TRACKING_PASSWORD", "")
    MLFLOW_EXPERIMENT_NAME:   str = "Holiday_Management_Agent"

    # ── Agent behaviour ──────────────────────────────────────────────────────
    MAX_AGENT_ROUNDS: int = 6   # maximum back-and-forth turns per planning session


settings = Settings()   # single shared instance — import this, not Settings()
'''

with open("holiday_management/config/settings.py", "w") as f:
    f.write(SETTINGS_CODE.strip())

print("✅ settings.py written")

✅ settings.py written


In [15]:
# ─────────────────────────────────────────────────────────────────────────────
# PHASE 5b — Shared State Object  (holiday_management/utils/state.py)
# ─────────────────────────────────────────────────────────────────────────────
# The State is passed like a baton along the agent pipeline.
# Each agent reads the relevant fields and writes its output to new fields.
# Using a dataclass + TypedDict keeps the state self-documenting.

STATE_CODE = '''
# state.py
# Defines the shared State dataclass that travels through every agent.
#
# Agent flow:
#   User Query ─→ [State.request] ─→ Planner ─→ [State.draft_plan]
#              ─→ Researcher ─→ [State.research_data]
#              ─→ final_output string

from dataclasses import dataclass, field
from typing import Optional, Dict, Any, List


@dataclass
class AgentState:
    """
    Central state object passed between every agent in the pipeline.

    Attributes
    ----------
    request        : Raw natural-language trip request from the user.
    user_id        : Identifier used for Mem0 personalisation lookup.
    draft_plan     : Skeleton itinerary produced by the Planner agent.
    research_data  : Verified facts dict produced by the Researcher agent.
    final_output   : Formatted Markdown travel guide (final deliverable).
    metadata       : Arbitrary metadata bag (token counts, timings, etc.).
    """

    request:       str
    user_id:       str = "anonymous"
    draft_plan:    Optional[str] = None
    research_data: Dict[str, Any] = field(default_factory=dict)
    final_output:  Optional[str] = None
    messages:      List[Dict[str, str]] = field(default_factory=list)
    metadata:      Dict[str, Any] = field(default_factory=dict)

    def add_message(self, role: str, content: str) -> None:
        """Append a message to the conversation history."""
        self.messages.append({"role": role, "content": content})

    def is_complete(self) -> bool:
        """True when the pipeline has produced a final itinerary."""
        return self.final_output is not None
'''

with open("holiday_management/utils/state.py", "w") as f:
    f.write(STATE_CODE.strip())

print("✅ state.py written")

✅ state.py written


In [16]:
# ─────────────────────────────────────────────────────────────────────────────
# PHASE 5c — Utility Helpers  (holiday_management/utils/utils.py)
# ─────────────────────────────────────────────────────────────────────────────

UTILS_CODE = '''
# utils.py
# General-purpose helpers shared across the holiday_management package.

import time
import logging
from typing import Any, Dict

logger = logging.getLogger(__name__)


def format_markdown_itinerary(raw_text: str, destination: str, days: int) -> str:
    """
    Wrap raw agent output in a consistent Markdown header.

    Parameters
    ----------
    raw_text    : The text produced by the final writing agent.
    destination : Name of the destination city/region.
    days        : Number of trip days.

    Returns
    -------
    str : Formatted Markdown string ready for display or export.
    """
    header = f"# 🌍 {days}-Day Travel Guide: {destination}\\n\\n"
    return header + raw_text


def timer(func):
    """
    Decorator that logs the wall-clock time of any function call.
    Useful for measuring LLM API latency.
    """
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        elapsed = time.time() - start
        logger.info("[TIMER] %s completed in %.2fs", func.__name__, elapsed)
        return result
    return wrapper


def safe_dict_get(d: Dict[str, Any], *keys, default=None) -> Any:
    """
    Safely traverse a nested dict with a list of keys.
    Returns `default` if any key is missing (avoids KeyError chains).
    """
    for key in keys:
        if not isinstance(d, dict):
            return default
        d = d.get(key, default)
    return d
'''

with open("holiday_management/utils/utils.py", "w") as f:
    f.write(UTILS_CODE.strip())

print("✅ utils.py written")

✅ utils.py written


---
## 🤖 Phase 6 — Agent Implementation (Groq LLM)

Two specialist agents form the core of the system:

| Agent | Role | Input | Output |
|---|---|---|---|
| **Planner** | Strategy layer | User request | Day-by-day skeleton itinerary |
| **Researcher** | Data layer | Skeleton itinerary | Verified facts & enriched plan |

Both use `langchain-groq` with `llama-3.1-8b-instant` — fast, free-tier friendly.


In [17]:
# ─────────────────────────────────────────────────────────────────────────────
# PHASE 6a — Planner Agent  (holiday_management/agents/planner.py)
# ─────────────────────────────────────────────────────────────────────────────
# The Planner is the STRATEGY layer.
# It receives the raw user request and builds a day-by-day "skeleton" itinerary
# that the Researcher will later enrich with verified facts.
#
# Why LangChain?
#   LangChain ≥1.2 provides:  prompt templates, chain composition,
#   output parsers, and a consistent interface across LLM providers.

PLANNER_CODE = '''
# planner.py
# Agent 1: The Planner (Strategy Layer)
#
# Responsibilities:
#   • Parse the user request for destination, duration, and interests
#   • Generate a geographically logical day-by-day skeleton itinerary
#   • Output plain-text that the Researcher can verify fact-by-fact
#
# LLM: Groq llama-3.1-8b-instant via LangChain ≥1.2

import os
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

from holiday_management.config.settings import settings
from holiday_management.utils.state import AgentState


# ── Prompt Template ────────────────────────────────────────────────────────
PLANNER_SYSTEM_PROMPT = """\
You are a senior travel strategist. Your job is to create a skeleton itinerary.

Rules:
- Organise activities geographically (nearby attractions on the same day).
- Keep Day 1 light — the traveller just arrived.
- Use bullet points. Each day gets 3-4 activity placeholders.
- Do NOT invent specific prices or opening times — the Researcher will verify those.
- Output ONLY the itinerary — no preamble, no sign-off.
"""

PLANNER_HUMAN_TEMPLATE = """\
User request: {request}

User preferences from memory: {user_memory}

Please create a {duration_days}-day skeleton itinerary for {destination}.
"""


def build_planner_chain():
    """Construct and return the LangChain planner chain."""
    print(f"[DEBUG] Planner LLM model: {settings.LLM_MODEL}, max_tokens: {settings.LLM_MAX_TOKENS}")
    # Initialise Groq LLM — token limit kept tight to respect free-tier quotas
    llm = ChatGroq(
        model=settings.LLM_MODEL,
        temperature=settings.LLM_TEMPERATURE,
        max_tokens=settings.LLM_MAX_TOKENS,
        api_key=settings.GROQ_API_KEY
    )

    prompt = ChatPromptTemplate.from_messages([
        ("system", PLANNER_SYSTEM_PROMPT),
        ("human",  PLANNER_HUMAN_TEMPLATE),
    ])

    # Chain: prompt → LLM → plain string output
    return prompt | llm | StrOutputParser()


def run_planner(state: AgentState,
                destination: str,
                duration_days: int,
                user_memory: str = "No prior preferences recorded.") -> AgentState:
    """
    Execute the Planner agent and store the skeleton itinerary in state.

    Parameters
    ----------
    state         : Shared agent state.
    destination   : Target travel destination.
    duration_days : Number of days in the trip.
    user_memory   : Retrieved Mem0 memory snippet for personalisation.

    Returns
    -------
    AgentState : Updated state with draft_plan populated.
    """
    chain = build_planner_chain()

    draft = chain.invoke({
        "request":       state.request,
        "destination":   destination,
        "duration_days": duration_days,
        "user_memory":   user_memory,
    })

    state.draft_plan = draft
    state.add_message("planner", draft)
    return state
'''

with open("holiday_management/agents/planner.py", "w") as f:
    f.write(PLANNER_CODE.strip())

print("✅ planner.py written")

✅ planner.py written


In [18]:
# ─────────────────────────────────────────────────────────────────────────────
# PHASE 6b — Researcher Agent  (holiday_management/agents/researcher.py)
# ─────────────────────────────────────────────────────────────────────────────
# The Researcher is the DATA VERIFICATION layer.
# It takes the Planner's skeleton and enriches each activity with
# simulated verified facts (addresses, hours, tips).
# In production, this agent would call real APIs (Google Places, Booking.com).

RESEARCHER_CODE = '''
# researcher.py
# Agent 2: The Researcher (Data Verification Layer)
#
# Responsibilities:
#   • Take the Planner\'s skeleton itinerary
#   • For each activity, simulate fact-checking (addresses, hours, tips)
#   • Return an enriched final itinerary in Markdown
#
# In production: swap the simulated facts with real API calls
# (Google Places API, TripAdvisor, Booking.com, etc.)

import os
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

from holiday_management.config.settings import settings
from holiday_management.utils.state import AgentState


# ── Prompt Template ────────────────────────────────────────────────────────
RESEARCHER_SYSTEM_PROMPT = """\
You are a meticulous travel fact-checker and content writer.

You receive a skeleton itinerary and must produce the FINAL, enriched travel guide.

Rules:
- For each activity, add realistic (but clearly labelled as approximate) details:
    * Best time to visit
    * Practical tip (e.g. book in advance, wear comfortable shoes)
    * Approximate cost bracket (budget / moderate / splurge)
- Use clean Markdown formatting with Day headers (## Day 1, ## Day 2, ...).
- End with a \"## Practical Tips\" section covering transport, language, currency.
- NEVER fabricate specific URLs or phone numbers.
- Be concise — max 80 words per day.
"""

RESEARCHER_HUMAN_TEMPLATE = """\
Destination: {destination}
Duration: {duration_days} days

Skeleton Itinerary (from Planner):
{draft_plan}

Please produce the enriched, final travel guide.
"""


def build_researcher_chain():
    """Construct and return the LangChain researcher chain."""
    print(f"[DEBUG] Researcher LLM model: {settings.LLM_MODEL}, max_tokens: {settings.LLM_MAX_TOKENS}")
    llm = ChatGroq(
        model=settings.LLM_MODEL,
        temperature=0.2,                  # lower temp for factual enrichment
        max_tokens=settings.LLM_MAX_TOKENS,
        api_key=settings.GROQ_API_KEY
    )

    prompt = ChatPromptTemplate.from_messages([
        ("system", RESEARCHER_SYSTEM_PROMPT),
        ("human",  RESEARCHER_HUMAN_TEMPLATE),
    ])

    return prompt | llm | StrOutputParser()


def run_researcher(state: AgentState,
                   destination: str,
                   duration_days: int) -> AgentState:
    """
    Execute the Researcher agent on the Planner\'s draft plan.

    Parameters
    ----------
    state         : Shared agent state (must have draft_plan populated).
    destination   : Target travel destination.
    duration_days : Number of trip days.

    Returns
    -------
    AgentState : Updated state with final_output populated.
    """
    if not state.draft_plan:
        raise ValueError("Researcher requires a draft_plan — run Planner first.")

    chain = build_researcher_chain()

    final = chain.invoke({
        "destination":   destination,
        "duration_days": duration_days,
        "draft_plan":    state.draft_plan,
    })

    state.final_output = final
    state.add_message("researcher", final)
    return state
'''

with open("holiday_management/agents/researcher.py", "w") as f:
    f.write(RESEARCHER_CODE.strip())

print("✅ researcher.py written")

✅ researcher.py written


---
## 🏃 Phase 7 — Team Orchestration & Full Pipeline Run

The `HolidayTeam` class wires the agents together in the correct order and exposes a single `.run()` entry point.


In [19]:
# ─────────────────────────────────────────────────────────────────────────────
# PHASE 7a — Holiday Team  (holiday_management/teams/holiday_team.py)
# ─────────────────────────────────────────────────────────────────────────────

TEAM_CODE = '''
# holiday_team.py
# The HolidayTeam orchestrates the full agent pipeline.
#
# Execution order:
#   1. Query Validation & Transformation
#   2. Pydantic model validation of the trip request
#   3. Mem0 memory retrieval (personalisation)
#   4. Planner Agent → draft_plan
#   5. Researcher Agent → final_output
#   6. Return AgentState

import sys
import os
import logging

from holiday_management.config.settings import settings
from holiday_management.utils.state import AgentState
from holiday_management.agents.planner import run_planner
from holiday_management.agents.researcher import run_researcher

logger = logging.getLogger(__name__)


class HolidayTeam:
    """
    Orchestrator that coordinates the Planner and Researcher agents.
    Also integrates Mem0 for personalised memory retrieval.
    """

    def __init__(self, enable_memory: bool = True):
        """
        Parameters
        ----------
        enable_memory : If True, retrieve past user preferences via Mem0
                        before planning.  Set False to skip (faster CI runs).
        """
        self.enable_memory = enable_memory
        self._mem = None

        if enable_memory:
            self._init_memory()

    def _init_memory(self):
        """Initialise Mem0 with Groq + ChromaDB."""
        print(f"[DEBUG] Mem0 init: using LLM model {settings.LLM_MODEL} with max_tokens {settings.LLM_MAX_TOKENS}")
        try:
            from mem0 import Memory
            config = {
                "vector_store": {
                    "provider": "chroma",
                    "config": {
                        "collection_name": "holiday_agent_memory",
                        "path": settings.CHROMA_DB_PATH
                    }
                },
                "llm": {
                    "provider": "groq",
                    "config": {
                        "model": settings.LLM_MODEL,
                        "temperature": 0,
                        "max_tokens": settings.LLM_MAX_TOKENS, # Use setting, not hardcoded 512
                        "api_key": os.environ.get("GROQ_API_KEY")
                    }
                },
                "embedder": {
                    "provider": "huggingface",
                    "config": {
                        "model": "all-MiniLM-L6-v2"
                    }
                }
            }
            self._mem = Memory.from_config(config)
            logger.info("Mem0 initialised successfully.")
        except Exception as e:
            logger.warning("Mem0 init failed (%s) — proceeding without memory.", e)
            self._mem = None

    def _get_user_memory(self, user_id: str, query: str) -> str:
        """Retrieve the top-1 relevant memory snippet for this user."""
        if self._mem is None:
            return "No prior preferences recorded."
        try:
            results = self._mem.search(query, filters={"user_id": user_id})
            memories = results.get("results") if isinstance(results, dict) else results
            if memories:
                return memories[0].get("memory", "No preferences.")
        except Exception as e:
            logger.warning("Memory search failed: %s", e)
        return "No prior preferences recorded."

    def run(self,
            request: str,
            destination: str,
            duration_days: int,
            user_id: str = "anonymous") -> AgentState:
        """
        Execute the full planning pipeline.

        Parameters
        ----------
        request       : Raw natural-language trip request.
        destination   : Target destination.
        duration_days : Number of trip days.
        user_id       : Used for Mem0 personalisation.

        Returns
        -------
        AgentState with final_output populated.
        """
        logger.info("[Team] Starting pipeline for: %s (%d days)", destination, duration_days)

        # Initialise shared state
        state = AgentState(request=request, user_id=user_id)

        # Step 1: retrieve user memory for personalisation
        user_memory = self._get_user_memory(user_id, request)
        logger.info("[Team] User memory: %s", user_memory)

        # Step 2: run Planner agent
        logger.info("[Team] Running Planner agent...")
        state = run_planner(state, destination, duration_days, user_memory)
        logger.info("[Team] Draft plan ready (%d chars)", len(state.draft_plan or ""))

        # Step 3: run Researcher agent
        logger.info("[Team] Running Researcher agent...")
        state = run_researcher(state, destination, duration_days)
        logger.info("[Team] Final output ready (%d chars)", len(state.final_output or ""))

        return state


# ── Singleton for direct import (mirrors original project pattern) ──────────
team = HolidayTeam(enable_memory=True)
'''

with open("holiday_management/teams/holiday_team.py", "w") as f:
    f.write(TEAM_CODE.strip())

print("✅ holiday_team.py written")

✅ holiday_team.py written


In [20]:
# ─────────────────────────────────────────────────────────────────────────────
# PHASE 7b — Run Full Pipeline & Save Output
# ─────────────────────────────────────────────────────────────────────────────
# This is the main integration test: send a travel request through both agents
# and display the final enriched itinerary.

import logging
logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s")

# Make sure the package root is on the path
import sys, os
import importlib
if '.' not in sys.path:
    sys.path.insert(0, '.')

# Force reload of modules to pick up latest settings
# Order matters: settings -> agents -> teams
import holiday_management.config.settings as settings_module
importlib.reload(settings_module)

import holiday_management.agents.planner as planner_module
importlib.reload(planner_module)

import holiday_management.agents.researcher as researcher_module
importlib.reload(researcher_module)

# Reload holiday_team *after* its dependencies (planner, researcher, settings)
import holiday_management.teams.holiday_team as team_module
importlib.reload(team_module)
from holiday_management.teams.holiday_team import HolidayTeam

# ── Run the pipeline ─────────────────────────────────────────────────────────
team = HolidayTeam(enable_memory=True)   # enable memory now that LLM is configured

state = team.run(
    request       = "I want to plan a trip to Paris for 5 days. I love art, food and walking.",
    destination   = "Paris",
    duration_days = 5,
    user_id       = "traveller_001"
)

# ── Display results ───────────────────────────────────────────────────────────
print("\n" + "="*60)
print("DRAFT PLAN (Planner Agent)")
print("="*60)
print(state.draft_plan)

print("\n" + "="*60)
print("FINAL ITINERARY (Researcher Agent)")
print("="*60)
print(state.final_output)

# ── Save outputs ──────────────────────────────────────────────────────────────
with open("phase_outputs/phase7_draft_plan.md", "w") as f:
    f.write(state.draft_plan or "")

with open("phase_outputs/phase7_final_itinerary.md", "w") as f:
    f.write(state.final_output or "")

print("\n✅ Phase 7 outputs saved:")
print("   phase_outputs/phase7_draft_plan.md")
print("   phase_outputs/phase7_final_itinerary.md")

[DEBUG] Mem0 init: using LLM model llama-3.1-8b-instant with max_tokens 256


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/usr/local/lib/python3.12/dist-packages/mem0/embeddings/huggingface.py:27: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.config.embedding_dims = self.config.embedding_dims or self.model.get_sentence_embedding_dimension()


[DEBUG] Mem0 init: using LLM model llama-3.1-8b-instant with max_tokens 256


...

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/usr/local/lib/python3.12/dist-packages/mem0/embeddings/huggingface.py:27: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.config.embedding_dims = self.config.embedding_dims or self.model.get_sentence_embedding_dimension()


[DEBUG] Planner LLM model: llama-3.1-8b-instant, max_tokens: 256


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


[DEBUG] Researcher LLM model: llama-3.1-8b-instant, max_tokens: 256

DRAFT PLAN (Planner Agent)
**Day 1: Arrival and Acclimatization**

* Arrive at Charles de Gaulle Airport
* Check-in to hotel and freshen up
* Explore local neighborhood (e.g. Le Marais, Montmartre) for dinner and atmosphere

**Day 2: Montmartre and the Latin Quarter**

* Visit Musée de Montmartre to learn about the area's artistic history
* Explore the winding streets and artist studios of Montmartre
* Wander through the charming streets of the Latin Quarter, stopping for coffee and croissants

**Day 3: Louvre and Île de la Cité**

* Visit the world-famous Louvre Museum to see the Mona Lisa and other masterpieces
* Cross the Seine to Île de la Cité and visit the stunning Notre-Dame Cathedral
* Explore the nearby Sainte-Chapelle, known for its stunning stained glass windows

**Day 4: Musée d'Orsay and the Seine River**

* Visit the Musée d'Orsay to see an impressive collection of Impressionist and Post-Impressionist ar

---
## 📊 Phase 8 — MLflow Experiment Tracking

Logs each pipeline run to MLflow on DagsHub (or locally) so you can:
- Compare prompt strategies across runs
- Track latency, token counts, and output quality metrics
- Store final itineraries as downloadable artifacts


In [21]:
# ─────────────────────────────────────────────────────────────────────────────
# PHASE 8 — MLflow Experiment Tracking
# ─────────────────────────────────────────────────────────────────────────────

import mlflow
import time
import os

from holiday_management.config.settings import settings

# ── Connect to MLflow tracking server ────────────────────────────────────────
mlflow.set_tracking_uri(os.environ.get("MLFLOW_TRACKING_URI", "./mlruns"))
mlflow.set_experiment(settings.MLFLOW_EXPERIMENT_NAME)

print(f"✅ MLflow connected → {os.environ.get('MLFLOW_TRACKING_URI')}")
print(f"   Experiment: {settings.MLFLOW_EXPERIMENT_NAME}")

# ── Log the pipeline run ──────────────────────────────────────────────────────
with mlflow.start_run(run_name="paris_5day_trip") as run:

    # ── Parameters: what settings produced this run ───────────────────────────
    mlflow.log_params({
        "model":           settings.LLM_MODEL,
        "max_tokens":      settings.LLM_MAX_TOKENS,
        "temperature":     settings.LLM_TEMPERATURE,
        "destination":     "Paris",
        "duration_days":   5,
        "memory_enabled":  False,
    })

    # ── Metrics: measurable quality signals ───────────────────────────────────
    draft_len  = len(state.draft_plan or "")
    output_len = len(state.final_output or "")

    mlflow.log_metrics({
        "draft_plan_chars":    draft_len,
        "final_output_chars":  output_len,
        "message_count":       len(state.messages),
    })

    # ── Artifacts: save the generated itinerary ───────────────────────────────
    mlflow.log_artifact("phase_outputs/phase7_draft_plan.md")
    mlflow.log_artifact("phase_outputs/phase7_final_itinerary.md")

    print(f"\n✅ MLflow run logged — Run ID: {run.info.run_id}")
    print(f"   draft_plan_chars  = {draft_len}")
    print(f"   final_output_chars= {output_len}")

2026/05/08 13:34:48 INFO mlflow.tracking.fluent: Experiment with name 'Holiday_Management_Agent' does not exist. Creating a new experiment.


✅ MLflow connected → https://dagshub.com/prithusarkar90/networksecurity.mlflow
   Experiment: Holiday_Management_Agent

✅ MLflow run logged — Run ID: baf9898630b44d41bc6d44c0ad14a696
   draft_plan_chars  = 1017
   final_output_chars= 1045
🏃 View run paris_5day_trip at: https://dagshub.com/prithusarkar90/networksecurity.mlflow/#/experiments/20/runs/baf9898630b44d41bc6d44c0ad14a696
🧪 View experiment at: https://dagshub.com/prithusarkar90/networksecurity.mlflow/#/experiments/20


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


---
## 🚀 Phase 9 — CI/CD: Git Push to GitHub

Commits and pushes the complete `holiday_management` package to:
[https://github.com/Prithu-Sarkar/GEN-AI-INDUSTRY-PROJECT_2](https://github.com/Prithu-Sarkar/GEN-AI-INDUSTRY-PROJECT_2)

> **Before running:** Add your GitHub PAT to Colab Secrets as `GITHUB_TOKEN`.


In [22]:
# ─────────────────────────────────────────────────────────────────────────────
# PHASE 9 — Git Configuration & Push to GitHub
# ─────────────────────────────────────────────────────────────────────────────
# Requires GITHUB_TOKEN in Colab Secrets.
# The .gitignore is written first to exclude Colab's kernel state files,
# __pycache__, compiled .pyc files, and the ChromaDB binary blobs.

import os
from google.colab import userdata

GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
REPO_URL     = f"https://{GITHUB_TOKEN}@github.com/Prithu-Sarkar/GEN-AI-INDUSTRY-PROJECT_2.git"
BRANCH       = "main"

# ── Write .gitignore  ─────────────────────────────────────────────────────────
GITIGNORE = """
# Python
__pycache__/
*.py[cod]
*.pyo
*.egg-info/
.eggs/
dist/
build/

# Colab / Jupyter kernel state (keeps notebook clean on GitHub)
.ipynb_checkpoints/
*.ipynb_checkpoints

# ChromaDB binary data (large, not needed in repo)
all-utils/db/

# Logs
*.log

# Environment / secrets
.env
.envrc

# MLflow local runs (tracked on DagsHub)
mlruns/
"""

with open(".gitignore", "w") as f:
    f.write(GITIGNORE.strip())

print("✅ .gitignore written")

✅ .gitignore written


In [ ]:
import subprocess, json, sys, shutil
import glob # Import glob for file pattern matching
import os

def run_cmd(cmd: str, check: bool = True) -> str:
    """Run a shell command and return stdout. Raises on non-zero exit if check=True."""
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if check and result.returncode != 0:
        print(f"STDERR: {result.stderr}")
        raise RuntimeError(f"Command failed: {cmd}")
    return result.stdout.strip()


def clean_notebook(path: str) -> None:
    """
    Strip volatile Colab metadata from the notebook before committing:
      - kernel_info / widget_state / id fields  (not visible in GitHub)
      - cell outputs and execution_count         (avoids large diffs)
    This keeps the committed file lean and prevents stale widget state
    from showing up as uncommitted changes on every run.
    """
    with open(path, "r") as f:
        nb = json.load(f)

    # Remove volatile top-level metadata
    for key in ["widget_state", "kernel_info", "language_info"]:
        nb.get("metadata", {}).pop(key, None)

    # Clear cell outputs and execution counts
    for cell in nb.get("cells", []):
        if cell.get("cell_type") == "code":
            cell["outputs"] = []
            cell["execution_count"] = None
        # Remove per-cell id fields that change every run
        cell.pop("id", None)

    with open(path, "w") as f:
        json.dump(nb, f, indent=1)
    print(f"✅ Notebook cleaned: {path}")


# ── Configure git identity ────────────────────────────────────────────────────
run_cmd('git config --global user.email "colab-ci@holiday-agent.ai"')
run_cmd('git config --global user.name "Holiday Agent CI"')

# ── Initialise repo if not already done ──────────────────────────────────────
if not os.path.exists(".git"):
    run_cmd("git init")
    run_cmd(f"git remote add origin {REPO_URL}")
    run_cmd(f"git checkout -b {BRANCH}")
    print("✅ Git repo initialised")
else:
    # Update remote URL in case token changed
    run_cmd(f"git remote set-url origin {REPO_URL}", check=False)
    print("✅ Git repo already initialised")

# ── Clean notebook before committing ─────────────────────────────────────────
# Dynamically find the current notebook name
current_notebook_name = None

# Try to get the path from COLAB_NOTEBOOK_PATH environment variable first
colab_notebook_path = os.environ.get('COLAB_NOTEBOOK_PATH')
if colab_notebook_path:
    current_notebook_name = os.path.basename(colab_notebook_path)
    print(f"Detected notebook file from COLAB_NOTEBOOK_PATH: {current_notebook_name}")
else:
    # Fallback to glob if COLAB_NOTEBOOK_PATH is not set
    notebook_files = glob.glob('*.ipynb')
    if notebook_files:
        current_notebook_name = notebook_files[0]
        print(f"Detected notebook file via glob: {current_notebook_name}")
    else:
        # As a last resort, use the name the user provided, but print a warning
        current_notebook_name = "Personalized_Holiday_Management_Agent.ipynb"
        print(f"Warning: Could not dynamically detect notebook name. Using user-provided name as fallback: {current_notebook_name}")

if not current_notebook_name:
    raise RuntimeError("Could not determine the current notebook's filename.")

clean_notebook(current_notebook_name)

# ── Stage, commit, push ───────────────────────────────────────────────────────
run_cmd("git add -A")
run_cmd('git commit -m "feat: Personalized Holiday Management Agent — Colab notebook"',
        check=False)   # check=False in case there\'s nothing new to commit
run_cmd(f"git push -u origin {BRANCH} --force")

print("\n🚀 Code pushed to GitHub successfully!")
print("   https://github.com/Prithu-Sarkar/GEN-AI-INDUSTRY-PROJECT_2")

---
## 📦 Phase 10 — Package All Outputs & Download

Zips all generated `.py` files, phase outputs, and the notebook into a single archive for download.


In [29]:
# ─────────────────────────────────────────────────────────────────────────────
# PHASE 10 — Zip & Download All Outputs
# ─────────────────────────────────────────────────────────────────────────────
# Creates a single archive containing:
#   • holiday_management/ package (all .py source files)
#   • all-utils/ utilities (.py files only — no binary ChromaDB)
#   • phase_outputs/ (text/markdown artefacts from each phase)
#   • This notebook (.ipynb)
#   • .gitignore

import zipfile
import os
from pathlib import Path
from google.colab import files

ARCHIVE_NAME = "Holiday_Agent_Colab_Outputs.zip"

# Patterns to include
INCLUDE_PATTERNS = [".py", ".ipynb", ".md", ".txt", ".json", ".log", ".gitignore"]
# Directories / patterns to skip
SKIP_DIRS = {"__pycache__", ".git", "all-utils/db", "mlruns"}


def should_include(path: Path) -> bool:
    """Return True if this file should be added to the archive."""
    # Skip binary / cache dirs
    for skip in SKIP_DIRS:
        if skip in str(path):
            return False
    return path.suffix in INCLUDE_PATTERNS or path.name == ".gitignore"


with zipfile.ZipFile(ARCHIVE_NAME, "w", zipfile.ZIP_DEFLATED) as zf:
    for fpath in Path(".").rglob("*"):
        if fpath.is_file() and should_include(fpath):
            zf.write(fpath, fpath)
            print(f"  + {fpath}")

print(f"\n✅ Archive created: {ARCHIVE_NAME}")

# ── Download to local machine ──────────────────────────────────────────────────
files.download(ARCHIVE_NAME)
print("📥 Download started!")

  + utility_logging_example.log
  + .config/.last_update_check.json
  + phase_outputs/phase3_logging_output.log
  + phase_outputs/phase7_final_itinerary.md
  + phase_outputs/phase2_query_transform_output.json
  + phase_outputs/phase7_draft_plan.md
  + phase_outputs/phase1_pydantic_output.txt
  + phase_outputs/phase4_mem0_output.txt
  + holiday_management/__init__.py
  + sample_data/README.md
  + sample_data/anscombe.json
  + .config/logs/2026.04.16/13.28.21.827521.log
  + .config/logs/2026.04.16/13.27.28.140888.log
  + .config/logs/2026.04.16/13.27.53.684484.log
  + .config/logs/2026.04.16/13.28.06.952902.log
  + .config/logs/2026.04.16/13.28.05.189231.log
  + .config/logs/2026.04.16/13.28.22.950479.log
  + all-utils/utilities/pydantic_models.py
  + all-utils/utilities/mem0_example.py
  + all-utils/utilities/query_validation_transformation.py
  + all-utils/utilities/logging_example.py
  + holiday_management/teams/__init__.py
  + holiday_management/teams/holiday_team.py
  + holiday_mana

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

📥 Download started!


---
## ✅ Summary

| Phase | Status | Output |
|---|---|---|
| 0 | ✅ | Packages installed, env vars set |
| 1 | ✅ | `pydantic_models.py` + `phase1_pydantic_output.txt` |
| 2 | ✅ | `query_validation_transformation.py` + `phase2_query_transform_output.json` |
| 3 | ✅ | `logging_example.py` + `phase3_logging_output.log` |
| 4 | ✅ | `mem0_example.py` + `phase4_mem0_output.txt` |
| 5 | ✅ | Full `holiday_management/` package |
| 6 | ✅ | `planner.py` + `researcher.py` (Groq LLM) |
| 7 | ✅ | `phase7_draft_plan.md` + `phase7_final_itinerary.md` |
| 8 | ✅ | MLflow run logged to DagsHub |
| 9 | ✅ | Pushed to GitHub |
| 10 | ✅ | `Holiday_Agent_Colab_Outputs.zip` downloaded |

---

## 🔑 Required API Keys

| Key | Required | Purpose |
|---|---|---|
| `GROQ_API_KEY` | ✅ **Yes** | Powers both Planner & Researcher agents and Mem0 memory reasoning |
| `MONGO_DB_URL` | ✅ **Yes** | MongoDB Atlas for persistent user data storage |
| `MLFLOW_TRACKING_URI` | ✅ **Yes** (if DagsHub) | Remote experiment tracking server URL |
| `MLFLOW_TRACKING_USERNAME` | ✅ **Yes** (if DagsHub) | DagsHub authentication |
| `MLFLOW_TRACKING_PASSWORD` | ✅ **Yes** (if DagsHub) | DagsHub token |
| `GITHUB_TOKEN` | ✅ **Yes** (Phase 9) | PAT for pushing to GitHub repo |
| `MEM0_API_KEY` | ⚠️ Optional | Only if using Mem0 cloud instead of local ChromaDB |
